In [1]:
full_path = "tau2_golden_gpt4o.log"

with open(full_path, "r") as f:
    data = f.read()

data = data.split("[INFO] TOOL ERROR STATISTICS:")[1:]

print(len(data))

50


In [2]:
data = [
    d.split("[INFO] Shutting down ReActAgent and closing event loop.")[0] for d in data
]

data = [d.split("[INFO]")[1].strip() for d in data]

data = [d.split("2026-")[0].strip() for d in data]

print(data[0])

{
  "raise_count_with_type": {},
  "error_calling_log": []
}


In [3]:
import json

tool_error_statistics = [json.loads(d) for d in data]

In [4]:
with open(full_path, "r") as f:
    data = f.read()

data = data.split("[INFO] Golden Evaluation Flag Counts:")[1:-1]

print(len(data))

50


In [5]:
data = [d.split("2026-")[0].strip() for d in data]

print(data[1])

{
  "pass": 6,
  "tool_call_raised_error": 1
}


In [6]:
golden_statistics = [json.loads(d) for d in data]

print(golden_statistics[0])

{}


In [8]:
res = {}

for i in range(50):
    tool_error_statistic = tool_error_statistics[i]
    golden_statistic = golden_statistics[i]
    original_error = tool_error_statistic["raise_count_with_type"]
    has_original_error = len(original_error) > 0
    golden_res = "pass"
    if golden_statistic.get("tool_call_raised_error", 0) > 0:
        golden_res = "definite_failure"
    elif golden_statistic.get("wrong_tool_argument", 0) > 0:
        golden_res = "potential_failure"
    elif golden_statistic.get("different_tool_called", 0) > 0:
        golden_res = "potential_failure"
    elif golden_statistic.get("no_tool_call", 0) > 0:
        golden_res = "missing_info"
    res_str = "original_error" if has_original_error else "no_original_error"
    res_str += f",{golden_res}"
    res[res_str] = res.get(res_str, 0) + 1
print(json.dumps(res, indent=4))

{
    "no_original_error,pass": 25,
    "no_original_error,definite_failure": 20,
    "original_error,definite_failure": 5
}
